In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [38]:
from sklearn.utils import resample
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

In [10]:
customer = pd.read_csv("C:\\Users\\piyum\\Documents\\python codes\\customer_data.csv")
customer = customer.drop(['fea_6', 'fea_11'], axis=1)

payment = pd.read_csv("C:\\Users\\piyum\\Documents\\python codes\\payment_data.csv")
payment = payment.drop(['prod_limit', 'report_date', 'update_date', 'prod_code'], axis=1)

In [12]:
payment_avg = payment.groupby('id').mean()
combined_data = pd.merge(payment_avg, customer, on='id').drop(['id'], axis=1)

In [14]:
# Fill missing values with mean column values
combined_data.fillna(combined_data.mean(), inplace=True)
combined_data = combined_data.round(decimals=0)

In [16]:
def create_dummies(df, columns):
    for col in columns:
        dummies = pd.get_dummies(df[col], drop_first=True)
        df = pd.concat([df, dummies], axis=1).drop(col, axis=1)
    return df

categorical_cols = ['fea_1', 'fea_3', 'fea_5', 'fea_7', 'fea_9']
combined_data = create_dummies(combined_data, categorical_cols)

In [18]:
X = combined_data.drop('label', axis=1)
y = combined_data['label']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

In [22]:
# Oversample the minority class in training data
train_data = pd.concat([X_train, y_train], axis=1)
low_credit_risk = train_data[train_data['label'] == 0]
high_credit_risk = train_data[train_data['label'] == 1]

high_credit_risk_upsampled = resample(high_credit_risk, 
                                      replace=True, 
                                      n_samples=len(low_credit_risk), 
                                      random_state=1)

balanced_train = pd.concat([low_credit_risk, high_credit_risk_upsampled])

X_train_balanced = balanced_train.drop('label', axis=1)
y_train_balanced = balanced_train['label']

In [26]:
# Ensure all column names are strings 
X_train_balanced.columns = X_train_balanced.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

In [28]:
scaler = StandardScaler()
X_train_balanced = scaler.fit_transform(X_train_balanced)
X_test = scaler.transform(X_test)


In [40]:
model = Sequential([
    # Input layer with batch normalization
    Dense(128, input_dim=X_train_balanced.shape[1], activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.4),

    # Hidden layer 1
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.3),

    # Hidden layer 2
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.3),

    # Output layer
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile the model with a custom learning rate for Adam
optimizer = Adam(learning_rate=0.0005) 

model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,489 (60.50 KB)

 Trainable params: 15,041 (58.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [42]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train_balanced, y_train_balanced, 
    validation_split=0.2, 
    epochs=50, 
    batch_size=32, 
    callbacks=[early_stopping]
)


Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.5016 - loss: 2.6860 - val_accuracy: 0.5765 - val_loss: 2.4237
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5528 - loss: 2.5080 - val_accuracy: 0.4745 - val_loss: 2.3805
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5768 - loss: 2.4116 - val_accuracy: 0.3608 - val_loss: 2.3453
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5687 - loss: 2.3591 - val_accuracy: 0.3451 - val_loss: 2.3099
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5869 - loss: 2.2585 - val_accuracy: 0.3765 - val_loss: 2.2450
Epoch 6/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5823 - loss: 2.1764 - val_accuracy: 0.3922 - val_loss: 2.1996
Epoch 7/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6208 - loss: 2.1518 - val_accuracy: 0.4039 - val_loss: 2.1411
Epoch 8/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6274 - loss: 2.0466 - val_accuracy: 0.4353 - val_loss

In [44]:
eval_results = model.evaluate(X_test, y_test)
print(f"Test Loss: {eval_results[0]}, Test Accuracy: {eval_results[1]}")

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7126 - loss: 0.9445 
Test Loss: 0.9481992125511169, Test Accuracy: 0.7130177617073059
